# Solution: Ring all-reduce

У каждого ранка хранится список чанков. В reduce-scatter ранк передаёт переносимый чанк соседу справа, а полученный слева чанк складывает со своей локальной частью. После `N - 1` шагов на ранке есть один полный чанк. All-gather переносит эти полные чанки тем же кольцом.

In [ ]:
from collections.abc import Callable

import torch


def ring_allreduce(
    rank_tensors: list[torch.Tensor],
    on_send: Callable[[int, int], None] | None = None,
) -> list[torch.Tensor]:
    world_size = len(rank_tensors)
    if world_size == 0:
        return []

    # clone keeps the simulation independent from all input storage.
    chunks = [
        list(torch.tensor_split(tensor.clone(), world_size))
        for tensor in rank_tensors
    ]

    # Reduce-scatter: a received chunk is added to this rank's own part.
    for step in range(world_size - 1):
        sent = []
        for rank in range(world_size):
            chunk_index = (rank - step) % world_size
            destination = (rank + 1) % world_size
            if on_send is not None:
                on_send(rank, destination)
            sent.append(chunks[rank][chunk_index])

        for rank in range(world_size):
            chunk_index = (rank - step - 1) % world_size
            chunks[rank][chunk_index] = (
                chunks[rank][chunk_index] + sent[(rank - 1) % world_size]
            )

    # All-gather: forward each complete chunk until every rank has all of them.
    for step in range(world_size - 1):
        sent = []
        for rank in range(world_size):
            chunk_index = (rank + 1 - step) % world_size
            destination = (rank + 1) % world_size
            if on_send is not None:
                on_send(rank, destination)
            sent.append((chunk_index, chunks[rank][chunk_index]))

        for rank in range(world_size):
            chunk_index, chunk = sent[(rank - 1) % world_size]
            chunks[rank][chunk_index] = chunk

    return [torch.cat(rank_chunks, dim=0) / world_size for rank_chunks in chunks]


`tensor_split` сохраняет порядок и создаёт последний чанк меньшей длины, поэтому `torch.cat` восстанавливает исходную форму по первой оси. Callback вызывается на каждом из `N × (N - 1)` обменов в каждой фазе: всего `2N(N - 1)` раз.

In [ ]:
from torch_judge import check

check('ring_allreduce')
